In [1]:
# Instalação das dependências incluindo o suporte local a Hugging Face
!pip install -q -U \
    llama-index \
    llama-index-embeddings-huggingface \
    llama-index-llms-huggingface \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers \
    wget

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install -q pymupdf pypdf llama-index-readers-file

In [3]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

CUDA: True
GPU: Tesla T4
VRAM: 14.56317138671875 GB


In [6]:
import wget
import os

url = "https://raw.githubusercontent.com/buenofgustavo/projeto-sistemas-multiagentes/main/dados/entradas/v2/explicacoes.jsonl"
output_dir = "data"

# Cria a pasta 'data' se ela não existir
os.makedirs(output_dir, exist_ok=True)

# Faz o download do arquivo
file_path = os.path.join(output_dir, "explicacoes.jsonl")
if not os.path.exists(file_path):
    print("Baixando o arquivo...")
    wget.download(url, file_path)
    print("\n✅ Download concluído!")
else:
    print("✅ O arquivo já existe na pasta.")

Baixando o arquivo...

✅ Download concluído!


In [7]:
import json
from llama_index.core import Document

documents = []

with open(file_path, "r", encoding="utf-8") as arquivo:
    for linha in arquivo:
        if not linha.strip():
            continue

        registro = json.loads(linha)

        # Guarda os demais campos como metadados
        metadados = {
            chave: valor
            for chave, valor in registro.items()
            if chave != "explicacao_negocio"
        }

        documento = Document(
            id_=registro["id"],
            text=registro["explicacao_negocio"],
            metadata=metadados,
            # Usa apenas a explicação para gerar os embeddings
            excluded_embed_metadata_keys=list(metadados),
            # Código disponível nos metadados para consulta explícita
            excluded_llm_metadata_keys=["parametrizacao_gobr"],
        )

        documents.append(documento)

print(f"✅ Total de documentos carregados: {len(documents)}")

✅ Total de documentos carregados: 233


In [8]:
from llama_index.core import Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# LLM LOCAL

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

Settings.llm = HuggingFaceLLM(
    model_name=MODEL_NAME,
    tokenizer_name=MODEL_NAME,

    context_window=8192,
    max_new_tokens=1024,

    device_map="auto",

    model_kwargs={
        "dtype": torch.float16,
    },

    generate_kwargs={
        "temperature": 0.1,
        "do_sample": False,
    },
)

# EMBEDDING LOCAL

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    device="cuda",
    embed_batch_size=16,
)


print("✅ LLM local configurado")
print("   Qwen2.5-3B-Instruct 4-bit")

print("✅ Embedding local configurado")
print("   BAAI/bge-m3")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ LLM local configurado
   Qwen2.5-3B-Instruct 4-bit
✅ Embedding local configurado
   BAAI/bge-m3


In [9]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

# 1. Configura o fatiador globalmente
# chunk_size: tamanho em tokens/palavras de cada trecho
# chunk_overlap: sobreposição para evitar que o contexto se perca entre quebras
Settings.text_splitter = SentenceSplitter(
    chunk_size=512,      # Tamanho recomendado para modelos padrão de embedding
    chunk_overlap=64     # Sobreposição de 64 tokens entre chunks consecutivos
)

print("Criando o índice de vetores na memória RAM...")
# O processo de indexação calcula os vetores e os mantém na memória
index = VectorStoreIndex.from_documents(documents)

print("✅ Índice criado com sucesso e mantido na memória RAM!")

Criando o índice de vetores na memória RAM...
✅ Índice criado com sucesso e mantido na memória RAM!


In [10]:
retriever = index.as_retriever(similarity_top_k=5)

pergunta = "Quais regras tratam da frequência de visitas aos clientes?"
resultados = retriever.retrieve(pergunta)

print(f"Chunks encontrados: {len(resultados)}")

for i, resultado in enumerate(resultados, start=1):
    print(f"\n--- Resultado {i} | Score: {resultado.score} ---")
    print("Metadados:", resultado.node.metadata)
    print(resultado.node.get_content())

Chunks encontrados: 5

--- Resultado 1 | Score: 0.6821575781367474 ---
Metadados: {'id': 'regra_153', 'arquivo_origem': 'regra_153.gobr', 'parametrizacao_gobr': '\n### Última solicitação: https://3.basecamp.com/3199393/buckets/26567206/card_tables/cards/8926424812\nversion 1.0\n\n#\n## Definição de times\n\ntime "time televendas":\n    vendedores todostelevendas\n\n#\n## Definição de carteiras\n\ncarteira "carteira televendas":\n    clientes carteirafixa("time televendas")\n    \n#\n## Parâmetros de planejamento\n\nparametros "Não agendar clientes DH com crédito igual ou menor que R$0,00 (televendas)":\n    da carteira("carteira televendas")\n    quando nao tem credito e tipo de cobranca igual("DH")\n    nao agendar\n\nparametros "Não agendar clientes BK com crédito menor que R$200,00 (televendas)":\n    da carteira("carteira televendas")\n    quando com menos de R$ 80 de credito disponivel e tipo de cobranca igual("BK")\n    nao agendar\n\nparametros "Não agendar clientes inadimplente